# Project: Predicting Incident Inflow from Reopen Rates

## 1️⃣ Business Context: Why This Matters

**The Problem:** Support teams often realize they are understaffed only *after* the queue explodes. A major driver of this "unexpected" volume is actually **reopen behavior** from previous weeks. When tickets are solved poorly, they bounce back, causing a ripple effect of new re-work and related incidents.

**The Solution:** By using **Linear Regression**, we can predict next week's **Incident Inflow** based on leading indicators like **Reopen Rate** and **Avg Resolution Time**.

### Efficiency Gains
1.  **Proactive Staffing:** Forecast spikes 1-2 weeks early and adjust shifts (L2/L3 support).
2.  **Smarter Routing:** Identify high-risk categories (e.g., VPN issues) and route them to senior agents immediately.
3.  **Cost Control:** Quantify the hidden cost of poor resolutions (e.g., "Current reopen rate will cost us 200 extra tickets next month").

## 2️⃣ Setup and Data Simulation
We will simulate 1 year (52 weeks) of Service Desk data associated with a specific Assignment Group (e.g., "Network Support").

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Style for premium visuals
sns.set(style="whitegrid", context="talk")
%matplotlib inline

# --- Simulate Data ---
np.random.seed(42)
weeks = 52

# Features (Independent Variables)
# 1. Avg Reopen Rate (%) per week: usually fluctuates between 5% and 25%
reopen_rate = np.random.uniform(5, 25, weeks)

# 2. Avg Resolution Time (Hours): fluctuates between 4 and 48 hours
resolution_time = np.random.uniform(4, 48, weeks)

# 3. Change Requests (Count): Number of system changes deployed that week
change_requests = np.random.randint(0, 10, weeks)

# Target (Dependent Variable): Next Week's Incident Inflow
# Base inflow is 500 tickets. 
# + 20 tickets for every 1% increase in Reopen Rate (Major driver)
# + 5 tickets for every extra hour of resolution time (Sluggishness builds backlog)
# + 15 tickets for every Change Request (Change-related incidents)
# + Random noise
inflow_volume = 500 + (20 * reopen_rate) + (5 * resolution_time) + (15 * change_requests) + np.random.normal(0, 50, weeks)

# Create DataFrame
df = pd.DataFrame({
    'Week': range(1, weeks + 1),
    'Reopen_Rate_Pct': reopen_rate,
    'Avg_Res_Time_Hours': resolution_time,
    'Change_Requests': change_requests,
    'Incident_Inflow': inflow_volume
})

df.head()

## 3️⃣ Exploratory Data Analysis (EDA)
Let's verify our hypothesis: **Does high Reopen Rate correlate with high Incident Inflow?**

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Reopen_Rate_Pct', y='Incident_Inflow', data=df, s=100, color='crimson')
plt.title('Impact of Reopen Rate on Incident Inflow', fontsize=20)
plt.xlabel('Avg Reopen Rate (%)')
plt.ylabel('Weekly Incident Inflow')
plt.show()

## 4️⃣ Model Training
We will build a Linear Regression model to quantify *exactly* how much each factor contributes to the workload.

In [ ]:
# Prepare features and target
X = df[['Reopen_Rate_Pct', 'Avg_Res_Time_Hours', 'Change_Requests']]
y = df['Incident_Inflow']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Model
model = LinearRegression()
model.fit(X_train, y_train)

print("Model Trained successfully!")

## 5️⃣ What We Learn from the Data (Key Insights)
We can inspect the model's coefficients to understand the **root causes** of inflow spikes.

In [ ]:
coefficients = pd.DataFrame(model.coef_, X.columns, columns=['Impact Coefficient'])
intercept = model.intercept_

print(f"Baseline Inflow (Intercept): {intercept:.2f} tickets")
display(coefficients)

### Interpretation

1.  **Reopen Rate Impact**: If the coefficient is around 20, it means **for every 1% increase in reopens, we get ~20 extra tickets next week**.
    *   *Insight:* Speed alone doesn't help—quality matters more.
2.  **Resolution Time**: Longer resolution times create backlogs.
3.  **Change Requests**: Validates that changes create ripple effects (3-5 days later).

The model gives managers a concrete number to justify training or process changes.

## 6️⃣ Actionable Controls: What Can You Do?

Based on these predictions, teams can implement the following **Efficiency Controls**:

### 🛑 Prevent Reopens at Source
-   **Enforce First-Time Fix:** require a "Resolution Checklist" before closing tickets.
-   **Smart Closure Rules:** If the model predicts high risk for a specific category (e.g., VPN), delay auto-closure and trigger peer review.

### 🧑‍🏫 Targeted Agent Training
-   Don't just train everyone. Focus on agents with high personal reopen rates in the identified high-risk categories.

### 🔁 Knowledge Management
-   If specific categories drive the inflow spikes, audit the Knowledge Base articles for those topics. Outdated articles lead to bad fixes and reopens.

In [ ]:
# Evaluate Accuracy
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)

print(f"Model R-Squared: {r2:.2f}")
print("An R2 close to 1.0 means our inputs (Reopens, Changes) strongly predict the Inflow.")